In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BertTokenizer, GPT2LMHeadModel
from peft import PeftModel

In [ ]:
BASE = "uer/gpt2-distil-chinese-cluecorpussmall"
ADAPTER_DIR = "./ft_out/checkpoint-50000"
tokenizer = BertTokenizer.from_pretrained(BASE, use_fast=True)
base_model = GPT2LMHeadModel.from_pretrained(BASE)
if torch.cuda.is_available():
    base_model = base_model.to("cuda")

In [ ]:
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model = model.merge_and_unload()   # 关键：把 LoRA 合并回 base 权重
model.eval()

In [ ]:
gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

print(gen("我叫吴静", max_new_tokens=120)[0]["generated_text"].replace(" ", ""))

In [20]:
from transformers import BertTokenizer, GPT2LMHeadModel, TextGenerationPipeline
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
from transformers import BertTokenizer, GPT2LMHeadModel, TextGenerationPipeline
tokenizer = BertTokenizer.from_pretrained("uer/gpt2-large-chinese-cluecorpussmall")
model = GPT2LMHeadModel.from_pretrained("uer/gpt2-large-chinese-cluecorpussmall")
text_generator = TextGenerationPipeline(model, tokenizer)
text_generator("今天星期一", max_length=100, do_sample=True)

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '今天星期一 不 是 周 末 ， 人 不 多 。 服 务 员 很 少 ， 大 概 是 因 为 周 末 。 牛 排 和 饭 的 味 道 不 错 ， 不 过 ， 分 量 实 在 是 太 少 了 。 这 样 的 价 钱 ， 我 宁 愿 去 吃 豪 客 来 了 。'}]

In [1]:
import torch.nn as nn
from transformers import AutoModelForCausalLM
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

model = AutoModelForCausalLM.from_pretrained("uer/gpt2-distil-chinese-cluecorpussmall")

mlp0 = model.transformer.h[0].mlp
act = mlp0.act

print("act object:", act)
print("act type:", type(act))
print("is nn.Module:", isinstance(act, nn.Module))
print("callable:", callable(act))

# 如果是函数，通常会有 __name__
print("act name:", getattr(act, "__name__", None))
print("act module:", getattr(act, "__module__", None))


act object: NewGELUActivation()
act type: <class 'transformers.activations.NewGELUActivation'>
is nn.Module: True
callable: True
act name: None
act module: transformers.activations
